[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/practice/basic/02_math_and_statistics.ipynb)

# Luyện tập cơ bản 2 — Toán và thống kê cho ML

Đại số tuyến tính, đạo hàm và gradient descent, xác suất, ước lượng — xen kẽ, mỗi bài một kỹ năng dùng hằng ngày khi đọc và viết code ML.

Chương liên quan: 02 (toán cho ML), 05 (nền tảng ML).

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/practice/basic

In [ ]:
import numpy as np

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true

**Bài 1.** Viết `cosine_similarity(a, b)` = $\dfrac{a\cdot b}{\|a\|\|b\|}$.

In [ ]:
def cosine_similarity(a, b):
    ### START CODE HERE ### (≈ 2 dòng)
    # STARTER: return 0.0
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
    ### END CODE HERE ###

In [ ]:
check_close(cosine_similarity([1, 0], [0, 1]), 0.0, message="vuông góc")
check_close(cosine_similarity([1, 2], [2, 4]), 1.0, message="cùng hướng")
check_close(cosine_similarity([1, 2, 3], [-1, -2, -3]), -1.0, message="ngược hướng")

**Bài 2.** Giải hệ $Ax = b$ với `A`, `b` cho sẵn bằng `np.linalg.solve` → `x`; so với cách dùng nghịch đảo — vì sao `solve` được khuyên dùng?

In [ ]:
A = np.array([[3.0, 1, 2], [1, 4, 1], [2, 1, 5]])
b = np.array([1.0, 2, 3])
### START CODE HERE ### (≈ 1 dòng)
x = np.linalg.solve(A, b)
### END CODE HERE ###

In [ ]:
check_close(A @ x, b, message="Ax = b")

`solve` dùng phân rã LU — nhanh hơn và **ổn định số** hơn tính $A^{-1}$ rồi nhân (tính nghịch đảo khuếch đại sai số làm tròn khi A gần suy biến).

**Bài 3.** Trị riêng và vector riêng của ma trận đối xứng `S` bằng `np.linalg.eigh` → `values` (tăng dần), `vectors`. Kiểm tra $Sv = \lambda v$ và các vector riêng trực chuẩn.

In [ ]:
S = np.array([[4.0, 1], [1, 3]])
### START CODE HERE ### (≈ 1 dòng)
values, vectors = np.linalg.eigh(S)
### END CODE HERE ###
print(values)

In [ ]:
check_close(S @ vectors, vectors * values, message="S v = λ v")
check_close(vectors.T @ vectors, np.eye(2), message="trực chuẩn")
check_close(values.sum(), np.trace(S), message="tổng trị riêng = vết")

**Bài 4.** Kiểm tra gradient bằng sai phân trung tâm: với $f(w) = \|Xw - y\|^2$, gradient giải tích $\nabla f = 2X^\top(Xw - y)$. Viết `numerical_grad(f, w, h=1e-6)` (sai phân trung tâm từng toạ độ) và so sánh.

In [ ]:
rng = np.random.default_rng(0)
X, y, w0 = rng.normal(size=(20, 3)), rng.normal(size=20), rng.normal(size=3)


def f(w):
    return float(np.sum((X @ w - y) ** 2))


def numerical_grad(f, w, h=1e-6):
    ### START CODE HERE ### (≈ 5 dòng)
    # STARTER: return np.zeros_like(w)
    grad = np.zeros_like(w, dtype=float)
    for i in range(len(w)):
        e = np.zeros_like(w, dtype=float)
        e[i] = h
        grad[i] = (f(w + e) - f(w - e)) / (2 * h)
    return grad
    ### END CODE HERE ###

In [ ]:
check_close(numerical_grad(f, w0), 2 * X.T @ (X @ w0 - y), tol=1e-4, message="gradient số khớp gradient giải tích")

**Bài 5.** Gradient descent tối thiểu $g(x, y) = (x - 3)^2 + 2(y + 1)^2$ từ $(0, 0)$, learning rate 0.1, 200 bước → `point`. Learning rate nào (trong 0.1, 0.4, 0.6) làm thuật toán **phân kỳ**? → `diverging_lr`.

In [ ]:
def gradient(p):
    return np.array([2 * (p[0] - 3), 4 * (p[1] + 1)])


def run_gd(lr, steps=200):
    ### START CODE HERE ### (≈ 4 dòng)
    # STARTER: return np.zeros(2)
    p = np.zeros(2)
    for _ in range(steps):
        p = p - lr * gradient(p)
    return p
    ### END CODE HERE ###


point = run_gd(0.1)
### START CODE HERE ### (≈ 1 dòng)
diverging_lr = [lr for lr in (0.1, 0.4, 0.6) if not np.all(np.isfinite(run_gd(lr))) or np.abs(run_gd(lr)).max() > 1e6]
### END CODE HERE ###
print(point, diverging_lr)

In [ ]:
check_close(point, [3, -1], tol=1e-6, message="hội tụ về (3, −1)")
check_equal(diverging_lr, [0.6], "lr > 2/L (L = 4 theo trục y) → phân kỳ")

**Bài 6.** Mô phỏng 100.000 lần tung hai xúc xắc (`rng = np.random.default_rng(0)`): `p_seven` = tỉ lệ tổng bằng 7. Sau đó tính chính xác $P(\text{bệnh}\mid +)$ với tỉ lệ mắc 2%, độ nhạy 95%, độ đặc hiệu 90% → `posterior`.

In [ ]:
rng = np.random.default_rng(0)
### START CODE HERE ### (≈ 3 dòng)
rolls = rng.integers(1, 7, size=(100_000, 2))
p_seven = float(np.mean(rolls.sum(axis=1) == 7))
posterior = 0.02 * 0.95 / (0.02 * 0.95 + 0.98 * 0.10)
### END CODE HERE ###
print(p_seven, round(posterior, 4))

In [ ]:
check_close(p_seven, 1 / 6, tol=0.01, message="P(tổng = 7) ≈ 1/6")
check_close(posterior, 0.1624, tol=1e-3, message="Bayes: phần lớn kết quả dương tính là dương tính giả")

**Bài 7.** MLE của phân phối chuẩn: với mẫu `sample`, MLE của $\mu$ và $\sigma^2$ là trung bình và phương sai **chia n**. Tính `mu_hat`, `var_hat`; và log-likelihood tại MLE `loglik` = $\sum \log \mathcal N(x_i;\hat\mu,\hat\sigma^2)$.

In [ ]:
sample = np.random.default_rng(3).normal(5, 2, 1000)
### START CODE HERE ### (≈ 3 dòng)
mu_hat = sample.mean()
var_hat = sample.var()
loglik = float(np.sum(-0.5 * np.log(2 * np.pi * var_hat) - (sample - mu_hat) ** 2 / (2 * var_hat)))
### END CODE HERE ###

In [ ]:
from scipy import stats  # noqa: E402

check_close([mu_hat, var_hat], [5, 4], tol=0.3, message="gần tham số thật")
check_close(loglik, stats.norm(mu_hat, np.sqrt(var_hat)).logpdf(sample).sum(), message="log-likelihood")
check_greater(loglik, stats.norm(mu_hat + 0.2, np.sqrt(var_hat)).logpdf(sample).sum(), "MLE cho likelihood lớn hơn tham số lệch")

**Bài 8.** Khoảng tin cậy bootstrap 95% (phân vị) cho **trung vị** của `incomes` (2.000 lần lấy mẫu lại, `rng = np.random.default_rng(1)`) → `ci = (lo, hi)`.

In [ ]:
incomes = np.random.default_rng(2).lognormal(np.log(15), 0.5, 300)
rng = np.random.default_rng(1)
### START CODE HERE ### (≈ 2 dòng)
medians = [np.median(rng.choice(incomes, len(incomes))) for _ in range(2000)]
ci = tuple(np.quantile(medians, [0.025, 0.975]))
### END CODE HERE ###
print(f"trung vị {np.median(incomes):.2f}, KTC 95% [{ci[0]:.2f}, {ci[1]:.2f}]")

In [ ]:
check_true(ci[0] < np.median(incomes) < ci[1], "KTC chứa trung vị mẫu")
check_less(ci[1] - ci[0], 3, "KTC đủ hẹp với 300 mẫu")